In [ ]:
import os
import math
import csv
import random
import numpy as np
import gymnasium as gym
import matplotlib.pyplot as plt

from time import sleep
from collections import deque, defaultdict
from itertools import count
from typing import Dict, Counter, Any


import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F

from importnb import Notebook
with Notebook():
    from LabLatencyModel import LatencyModel, MultiDULatencyModel
    from LabCacheEngine import CacheEngineEnv, CacheUnitMapper
    from LabUserTileRequest import UserTileRequestEvents
    from LabEnvWrapper import EnvWrapper

import sys
sys.path.append('/home/eduardo/Workspace/CacheVideoPredict360/Sources')
# sys.path.append(r'c:\Users\es25591\Workspace\CacheVideoPredict360\Sources')
# from Common.Utils import save_training_results


In [ ]:
# --- 1. CONFIGURATION & HYPERPARAMETERS (Section VII-B) ---
class Config:
    n_episodes: int = 300
    n_nodes: int = 3
    n_users: int = 200
    step_size: float = 10.0
    arrival_rate: float = 10.0  # users per second
    zipf_alpha: float = 1.0
    n_videos: int = 500
    n_gops: int = 30
    n_layers: int = 2
    n: int = 4
    m: int = 3
    n_tiles: int = n * m
    tiles_per_viewport: int = 4

    base_tile_b = 2e6 / n_tiles
    enh_tile_b = 15e6 / n_tiles

    max_capacity: float = 500e6  # 500 MB
    cache_capacity_percent: float = 0.05  # 5% of the total video size
    cache_size: int = int(n_videos * cache_capacity_percent)
    cache_capacity_b: float = (
        n_gops * n_tiles * base_tile_b +
        n_gops * tiles_per_viewport * enh_tile_b
    ) * cache_size

    # Hyperparameters for RL
    epsilon_start: float = 1.0
    epsilon_min: float = 0.005
    epsilon_decay: float = 0.987
    gamma: float = 0.99
    learning_rate: float = 1e-3
    batch_size: int = 32
    buffer_capacity: int = 2000
    window_len: int = 3  # LSTM sequence length (history window)
    nb_interval: int = 200  # train every 200 requests

    h_short: int = 300   # sliding windows for popularity (Section VI-A)
    h_long: int = 1000

    r_base: float = 30.0 # PSNR reward for base layer (Section VI-C)
    r_enh: float = 10.0  # PSNR reward for enhancement layer
    penalty: float = 0.0 # fetch penalty (implicit in paper)

    # CPT parameters
    theta: float = 0.5
    lam: float = 3.7183

    @property
    def state_dim(self) -> int: # 10*C + 2 = (2C + 2Ck) * 2 + 2 (Section VI-A)
        return 10 * self.cache_size + 2

    @property
    def action_dim(self) -> int: # |A| = 5C + 1 (Section VI-B)
        return (self.cache_size + self.cache_size * self.tiles_per_viewport + 1)

In [ ]:
def save_training_results(
    path_,
    filename,
    ep, 
    total_reward, 
    cache_hits, 
    cache_misses, 
    soft_hits
):
    with open(os.path.join(path_, filename), 'a', newline='') as f:
        fieldnames = [
            'episode', 
            'total_reward', 
            'cache_hits', 
            'cache_misses', 
            'soft_hits'
        ]
        writer_results = csv.DictWriter(f, fieldnames=fieldnames)

        if ep == 0:
            writer_results.writeheader()
        
        writer_results.writerow({
            'episode': ep,
            'total_reward': total_reward,
            'cache_hits': cache_hits,
            'cache_misses': cache_misses,
            'soft_hits': soft_hits
        })


In [ ]:
class FeatureAdapter:
    def __init__(self, env: CacheEngineEnv, cfg: Config):
        self.env = env
        self.cfg = cfg

        self.video_hist_short = deque(maxlen=cfg.h_short)
        self.video_hist_long = deque(maxlen=cfg.h_long)
        self.tile_hist_short = deque(maxlen=cfg.h_short)
        self.tile_hist_long = deque(maxlen=cfg.h_long)
        self.tiles_hist_short = deque(maxlen=cfg.h_short)
        self.tiles_hist_long = deque(maxlen=cfg.h_long)

        self.video_freq_short = defaultdict(int)
        self.video_freq_long = defaultdict(int)
        self.tile_freq_short = defaultdict(int)
        self.tile_freq_long = defaultdict(int)
        self.tiles_freq_short = defaultdict(int)
        self.tiles_freq_long = defaultdict(int)

    def reset_history(self):
        queues = (
            self.video_hist_short,
            self.video_hist_long,
            self.tiles_hist_short,
            self.tiles_hist_long,
            self.tile_hist_short,
            self.tile_hist_long,
        )
        freqs = (
            self.video_freq_short,
            self.video_freq_long,
            self.tiles_freq_short,
            self.tiles_freq_long,
            self.tile_freq_short,
            self.tile_freq_long,
        )
        
        for q in queues:
            q.clear()
        for f in freqs:
            f.clear()

    def update_history(self, vid: int, tiles: list[int]):       
        self._update_window(self.video_hist_short, self.video_freq_short, vid)
        self._update_window(self.video_hist_long, self.video_freq_long, vid)

        tiles = tuple(tiles) if tiles is not None else None

        if tiles is None:
            return

        self._update_window(self.tiles_hist_short, self.tiles_freq_short, tiles)
        self._update_window(self.tiles_hist_long, self.tiles_freq_long, tiles)

        for tile in tiles:
            self._update_window(self.tile_hist_short, self.tile_freq_short, (vid, tile))
            self._update_window(self.tile_hist_long, self.tile_freq_long, (vid, tile))

    def _update_window(self, hist_queue: deque, freq_dict: Dict, item):
        if len(hist_queue) == hist_queue.maxlen:
            old_item = hist_queue.popleft()
            freq_dict[old_item] -= 1
            if freq_dict[old_item] == 0:
                del freq_dict[old_item]
        hist_queue.append(item)
        freq_dict[item] += 1

In [ ]:
class LruPolicy:
    def __init__(self, max_items: int):
        self.max_items = max_items
        self.cache = set()        # store video ids only
        self.access_order = []    # video ids in LRU order

    def get(self, vid):
        if vid not in self.cache:
            return None
        self.access_order.remove(vid)
        self.access_order.append(vid)
        return vid

    def put(self, vid):
        evicted = []
        if vid in self.cache:
            self.access_order.remove(vid)
        else:
            while len(self.cache) >= self.max_items and self.access_order:
                lru_vid = self.access_order.pop(0)
                self.cache.remove(lru_vid)
                evicted.append(lru_vid)

        self.cache.add(vid)
        self.access_order.append(vid)
        return evicted

    def contains(self, vid) -> bool:
        return vid in self.cache

    def remove(self, vid):
        if vid not in self.cache:
            return False

        self.access_order.remove(vid)
        self.cache.remove(vid)
        return True

    def clear(self):
        self.cache.clear()
        self.access_order.clear()

    def get_stats(self) -> Dict[str, Any]:
        return {
            'size': len(self.cache),
            'max_size': self.max_items,
            'num_items': len(self.cache),
            'utilization': len(self.cache) / self.max_items if self.max_items > 0 else 0
        }

    def keys(self):
        return self.cache.keys()

    def stats(self) -> Dict[str, Any]:
        return {
            'size': len(self.cache),
            'max_size': self.max_items,
            'num_items': len(self.cache),
            'utilization': len(self.cache) / self.max_items if self.max_items > 0 else 0
        }

In [ ]:
class NetworkAdapter:
    def __init__(self, env: EnvWrapper, feature_adapter: FeatureAdapter, cfg: Config):
        self.env = env
        self.cfg = cfg
        self.features = feature_adapter

        self.capacity = int(
            self.cfg.n_videos * self.cfg.cache_capacity_percent * self.cfg.n_gops * self.cfg.n_tiles +
            self.cfg.n_videos * self.cfg.cache_capacity_percent * self.cfg.n_gops * self.cfg.tiles_per_viewport            
        )
        self.n_features = self.capacity + 1

        self.C = self.cfg.cache_size               # paper’s cache capacity (videos)
        self.k = self.env.mec_cache.get_viewport_tile_budget()

        self.video_cache_index = [-1] * self.C
        self.tile_cache_index = [[-1] * self.k for _ in range(self.C)]
        
        self.policy = LruPolicy(self.C)
        self.video_order = []
        
        print(f"NetworkAdapter initialized with capacity: {self.C} videos, {self.k} tiles per video")

    def _cache_video(self, vid: int = 0, layer: int = 0):
        self.env.mec_cache.cache_new_video(vid, gop_id=0, layer=layer)

    def _evict_video(self, bitmap: np.ndarray, v: int):
        for layer, tile_id, gop_id in np.argwhere(bitmap[v] == 1):
            bitmap[v, layer, tile_id, gop_id] = 0
    
    def _evict_tile(self, bitmap: np.ndarray, vid: int, tile_id: int):
        for gop_id in np.where(bitmap[vid, 1, tile_id, :] == 1)[0]:
            bitmap[vid, 1, tile_id, gop_id] = 0

    def get_video_cache_idx(self, vid: int) -> int:
        for idx, v in enumerate(self.video_cache_index):
            if v == vid:
                return idx
        return -1

    def cache_video(self, vid: int):
        vid_idx = self.get_video_cache_idx(vid)
        if vid_idx != -1:
            self.video_order.remove(vid)
            self.video_order.append(vid)
            return 
        else:
            vid_evict = self.video_order.pop(0) if len(self.video_order) >= self.C else None
            vid_idx = self.get_video_cache_idx(vid_evict) if vid_evict is not None else -1
            if vid_evict is not None and vid_idx != -1:
                self.video_cache_index[vid_idx] = -1
                self.tile_cache_index[vid_idx] = [-1] * self.k
            self.video_order.append(vid)
            
        self.video_cache_index[vid_idx] = vid
        self.tile_cache_index[vid_idx] = [-1] * self.k
        
    def build_observation(self, request: Dict, bitmap: np.ndarray) -> np.ndarray:
        """
        Builds the state vector as in the paper:
          [ x_s (C), y_s (C*k), z_s (1), x_l (C), y_l (C*k), z_l (1) ]
        where:
          - x_s/x_l: counts of requests for cached base videos (short/long windows)
          - y_s/y_l: counts of requests for cached enh tiles per video (short/long)
          - z_s/z_l: counts for the currently examined item (video or tile)
        Total dim = 10*C + 2 when k=4.
        """
        C, k = self.C, self.k

        # Initialize feature blocks
        x_s = np.zeros(C, dtype=np.float32)
        x_l = np.zeros(C, dtype=np.float32)

        y_s = np.zeros(C * k, dtype=np.float32)
        y_l = np.zeros(C * k, dtype=np.float32)

        # Takes in consideration the ranked cached videos using the long-term popularity video
        
        for i, vid in enumerate(self.video_cache_index):
            if vid == -1:
                continue
            x_s[i] = self.features.video_freq_short.get(vid, 0)
            x_l[i] = self.features.video_freq_long.get(vid, 0)

            tiles = self.tile_cache_index[i]

            for j, t in enumerate(tiles):
                if t == -1:
                    continue
                y_s[i * k + j] = self.features.tile_freq_short.get((vid, t), 0)
                y_l[i * k + j] = self.features.tile_freq_long.get((vid, t), 0)

        vid, gop = request["video"], request["gop"]
        viewport = request["viewport"] if request["viewport"] is not None else []

        z_s = np.array([
            self.features.video_freq_short.get(vid, 0) if gop == 0
            else (sum(self.features.tile_freq_short.get((vid, t), 0) for t in viewport) / max(len(viewport), 1))
        ], dtype=np.float32)

        z_l = np.array([
            self.features.video_freq_long.get(vid, 0) if gop == 0
            else (sum(self.features.tile_freq_long.get((vid, t), 0) for t in viewport) / max(len(viewport), 1))
        ], dtype=np.float32)

        return np.concatenate([x_s, x_l, y_s, y_l, z_s, z_l], axis=0)
    
    def last_sample_replication(
        self, 
        vid: int, 
        gop: int, 
        viewport: list[int],
    ):
        for tile in viewport:
            self.env.mec_cache.cache_new_video(vid, gop, layer=1, tile_id=tile)
        
        vid_idx = self.get_video_cache_idx(vid)

        for i, tile in enumerate(viewport):
            self.tile_cache_index[vid_idx][i] = tile

    def calc_cache_hits(
        self, 
        vid: int, 
        viewport: list[int], 
    ) -> tuple[int, float]:
        hits = 0
        distortion = 0.0

        vid_idx = self.get_video_cache_idx(vid)
        tiles_idx = self.tile_cache_index[vid_idx] if vid_idx != -1 else []

        if vid_idx != -1:
            hits += 12  # base layer hit
            distortion += 30.0
        else:
            return hits, distortion
        
        for t_idx in viewport:
            if vid_idx != -1 and t_idx in tiles_idx:
                hits += 1  # enhancement layer hit
                distortion += 2.5

        return hits, distortion
    
    def get_next_user_gop(self, u: int, gop: int, cfg: Config) -> list[int]:
        if gop + 1 >= cfg.n_gops:
            return np.array([], dtype=int)
        
        current_viewport = self.env.users_env.users_viewport_tiles[u][gop+1]
        return np.array(
            [y * cfg.n + x for x, y in current_viewport], dtype=int
        )


In [ ]:
if __name__ == "__main__":
    print("--- Starting DRL Caching System ---")

    # 1. Load Configuration
    cfg = Config()

    # 2. Initialize Environment
    du_caches = []

    unit_mapper = CacheUnitMapper(
        cache_capacity_mb=cfg.cache_capacity_b / 1e6,
        num_gops=cfg.n_gops,
        num_tiles=cfg.n_tiles,
        viewport_tiles=4,  # assuming viewport with 4 tiles
        base_tile_mb=2e6 / 1e6 / cfg.n_tiles,
        enh_tile_mb=15e6 / 1e6 / cfg.n_tiles
    )

    mec_cache = CacheEngineEnv(
        n_users=cfg.n_users,
        n_videos=cfg.n_videos,
        n_layers=cfg.n_layers,
        n_tiles=cfg.n_tiles,
        n_gops=cfg.n_gops,
        cache_capacity=cfg.cache_capacity_b,
        # policy=SvcLruPolicy(max_size=max_capacity)
        unit_mapper=unit_mapper
    )

    # Initialize User Environment
    users_env = UserTileRequestEvents(
        n_nodes=cfg.n_nodes,
        n_users=cfg.n_users,
        n_videos=cfg.n_videos,
        n_gops=cfg.n_gops,
        n_layers=cfg.n_layers,
        n_tiles=cfg.n_tiles,
        n=cfg.n,
        m=cfg.m,
        users_viewport_tiles=None,
        requested_videos=None,
        users_arrivals=None,
        arrival_rate=cfg.arrival_rate,
        alpha=cfg.zipf_alpha
    )

    P = cfg.n_nodes; max_U = cfg.n_users
    lat_model = MultiDULatencyModel(
        P=P, 
        max_U=max_U,
        R_M_D=80e6,  # 640 Mbps -> 80e6 B/s
        R_C_M=125e6, # 1 Gbps -> 125e6 B/s
        mu=2e7, 
        eta=2e5,
        B_pu_matrix=np.full((P, max_U), 20e6, dtype=float),    # 160 Mbps -> 20e6 B/s
        gamma_pu_matrix=np.full((P, max_U), 5.0, dtype=float), # SNRs
        rhoT_p=[0.2], 
        lambda_p=[0.05],
        du_fixed_delay=0.001,  # 1 ms
        mec_fixed_delay=0.005, # 5 ms
        cloud_fixed_delay=0.1  # 100 ms
    )

    env = EnvWrapper(
        n=cfg.n,
        n_layers=cfg.n_layers,
        users_env=users_env, 
        du_caches=du_caches,
        mec_cache=mec_cache,
        latency_model=lat_model,
        theta=cfg.theta,
        lam=cfg.lam
    )

    obs, info = env.reset()
    feature_adapter = FeatureAdapter(env, cfg)
    net_adapter = NetworkAdapter(env, feature_adapter, cfg)

    for episode in range(cfg.n_episodes):

        obs, info = env.reset()
        feature_adapter.reset_history()
        
        cache_hits = 0
        cache_misses = 0
        soft_hits = 0.0
        total_reward = 0.0
        avg_psnr = []
        
        for step in count():

            # Get active users (not finished all GOPs)
            reqs_state = info['users_requests']
            active_users = [
                req for req in reqs_state if req['gop'] < cfg.n_gops
            ]

            # Count active users per DU
            active_users_per_du = Counter(req["p"] for req in active_users)

            # Get current cache bitmap ( Videos x Layers x Tiles x GOPs )
            bitmap = net_adapter.env.mec_cache.get_cache_bitmap()

            # Main Loop. Process each active user request
            for req in active_users:
                u, p, v, g, tiles = req['u'], req['p'], req['video'], req['gop'], req["tiles"]
                viewport = req['viewport'] if req['viewport'] is not None else []

                has_base_layer = not np.all(bitmap[v, 0, :, :] == 0)
                if g == 0 and not has_base_layer:
                    net_adapter.cache_video(vid=v)

                    net_adapter.last_sample_replication(v, g, viewport)
                elif g > 0 and has_base_layer:
                    net_adapter.last_sample_replication(v, g, viewport)
                
                # Compute Cache Performance
                next_viewport = net_adapter.get_next_user_gop(u, g, cfg)

                ch, distortion = net_adapter.calc_cache_hits(v, next_viewport)

                cache_hits += ch
                cache_misses += len(tiles) - ch
                soft_hits += float(ch) / 16.0 if len(tiles) > 0 else 0.0

                print(
                    f"Step {step}, User {u}, "
                    f"Video: {v}, Gop: {g}, Viewport: {viewport} \n"
                    f"Cache Hits: {cache_hits}, "
                    f"Misses: {cache_misses}, "
                    f"Soft Hits: {soft_hits:.2f}, "
                    f"Current Cache Hit: {ch}"
                    # f"Hit Ratio: {hit_ratio:.2f}\n"
                )

            # Updates in the state after processing all active users
            reqs_next_state = net_adapter.env.users_env.step(
                None, bitmap
            )

            done = (net_adapter.env.users_env.users_done >= net_adapter.env.users_env.n_users)
            if done:
                break

            info = {
                'users_requests': reqs_next_state
            }

            print(f"Step {step}, Active Users: {len(active_users)}")
            # print(f"Request State: {reqs_state}")
            # print(f"Next Request State: {reqs_next_state}")
            print("----------------------------------------------------------------")
        
        filename = (
            f"lsr_E{cfg.n_episodes}_U{cfg.n_users}_"
            f"g{cfg.gamma}_"
            f"V{cfg.n_videos}_G{cfg.n_gops}_L{cfg.n_layers}_n{cfg.n}_m{cfg.m}_"
            f"cap{cfg.cache_size}_AR{cfg.arrival_rate}_Z{cfg.zipf_alpha}.csv"
        )

        save_training_results(
            path_='/home/eduardo/Workspace/CacheVideoPredict360/Results',
            # path_=r'c:\Users\es25591\Workspace\CacheVideoPredict360\Results',
            filename=filename,
            ep=episode,
            total_reward=total_reward,
            cache_hits=cache_hits,
            cache_misses=cache_misses,
            soft_hits=soft_hits
        )

In [ ]:
import itertools
import numpy as np

def build_action_space(C: int, k: int):
    # A1: base-not-cached actions (0 = no-op, 1..C evict ranked i-th video)
    A1 = list(range(C + 1))
    
    # A2: enhance-tile actions (C*k entries, each replaces one tile position)
    A2 = list(range(C * k))
    
    # Cartesian product (indices)
    A = list(itertools.product(A1, A2))
    
    return A1, A2, A  # A has size (C+1) * (C*k) = 5C+1 when k=4

# Example usage
C, k = 2, 4  # cache can hold 2 videos; viewport budget k=4
A1, A2, A = build_action_space(C, k)
print("A1 (C+1):", A1)
print("A2 (C*k):", A2)
print("Total |A|:", len(A), "== (C+1)*C*k =", (C+1)*C*k)

# One-hot encoding helper for the flat index in [0, 5C]
def one_hot_action(idx: int, C: int, k: int):
    size = (C + 1) + C * k  # 5C+1 when k=4
    vec = np.zeros(size, dtype=np.float32)
    vec[idx] = 1.0
    
    return vec

# Example: pick A1 action i=1 (evict first video) and A2 action m=3 (replace tile slot 3)
a1_choice, a2_choice = 1, 3
flat_idx = a1_choice if a1_choice <= C else (C + 1 + a2_choice)
action_vec = one_hot_action(flat_idx, C, k)

print("Flat index:", flat_idx)
print("One-hot vector:", action_vec)